# L03 · SFT, off-policy, and on-policy

## Goal

**Estimated time:** 40 min · **Path:** fast, full

- identify who creates training states
- compare SFT and KD fairly
- audit response budgets

### Current position: L02 → **L03** → L04

```text
Prompt/Data -> state source -> ... -> L03 -> ... -> fair evaluation
```

Alt text: The course map highlights L03 between its prerequisite and next lesson; every method remains connected to the same evaluation stage.

## Setup

In [1]:
LESSON_ID = "L03"
from pathlib import Path
import sys
import torch

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root / "src"))

import opd_study
from opd_study.device import resolve_device
from opd_study.utils import seed_everything

seed_everything(42)
device_report = resolve_device("cpu")
print({"lesson": LESSON_ID, "opd_study": opd_study.__version__,
       "torch": torch.__version__, "device": device_report.selected,
       "profile": "toy", "network": "not required"})

{'lesson': 'L03', 'opd_study': '0.1.0.dev0', 'torch': '2.13.0', 'device': 'cpu', 'profile': 'toy', 'network': 'not required'}


## Steps

### 1/3 · 8–12 min

SFT and off-policy KD use different targets but learn on fixed demonstration prefixes. A fair comparison begins with the same student initialization and response-token budget.

Figure alt: labels and numbers remain readable without color.

### Core mechanics

In state-distribution notation, SFT/KD usually use `s ~ d_data`; OPD uses `s ~ d_student`. Error prefixes created at inference may be nearly absent from `d_data`. OPD exposes that covariate shift, but a very weak student may visit only states where teacher guidance is unstable.

A fair comparison fixes initial weights, prompt subset, optimizer, learning rate, response-token count, and evaluation split. Equal optimizer steps alone are insufficient because methods may process different numbers of tokens.

### Production implementation: why this design

`TrajectoryBatch` places token IDs, attention/response masks, prompt lengths, and rollout snapshots under one contract. SFT and KD use the same batch and response-target count; SFT reads token IDs while KD reads detached teacher logits.

Production code: [`types.py`](../../src/opd_study/types.py), [`core.py`](../../src/opd_study/training/core.py).

In [2]:
import inspect
from opd_study.algorithms import supervised_fine_tuning_loss, off_policy_kd_loss

objects_to_show = (supervised_fine_tuning_loss, off_policy_kd_loss,)
for object_to_show in objects_to_show:
    source_lines = inspect.getsource(object_to_show).splitlines()
    print(f"\n# {object_to_show.__module__}.{object_to_show.__qualname__}")
    print("\n".join(source_lines[:80]))
    if len(source_lines) > 80:
        print(f"... {len(source_lines) - 80} more lines; open the linked source file")


# opd_study.algorithms.losses.supervised_fine_tuning_loss
def supervised_fine_tuning_loss(
    student_logits: Tensor,
    trajectories: TrajectoryBatch,
) -> LossOutput:
    """Hard-label next-token cross entropy on demonstration response tokens."""

    shifted_logits, target_ids, _, prediction_mask = shifted_causal_tensors(
        student_logits,
        trajectories.token_ids,
        trajectories.attention_mask,
        trajectories.response_mask,
    )
    shifted_loss = cross_entropy_from_logits(shifted_logits, target_ids)
    loss = masked_mean(shifted_loss, prediction_mask)
    token_loss, effective_mask = _restore_token_alignment(
        shifted_loss, prediction_mask, trajectories.token_ids.shape[1]
    )
    return LossOutput(
        loss=loss,
        token_loss=token_loss,
        effective_mask=effective_mask,
        metrics=_metrics(loss, effective_mask, prefix="sft"),
    )

# opd_study.algorithms.losses.off_policy_kd_loss
def off_policy_kd_loss(
    student_logits

### Alternatives and trade-offs

State sources can mix per batch, example, token, or turn. The mini runtime chooses per batch for reproducibility and clarity. Finer mixing may reduce variance but complicates trajectory provenance.

### 2/3 · Run and observe

Predict before running: which invariant should you inspect first in L03's output? Write one sentence, then run.

In [3]:
import copy
from opd_study.algorithms import off_policy_kd_loss, score_teacher, supervised_fine_tuning_loss
from opd_study.data import CharacterTokenizer, collate_examples, generate_tiny_arithmetic
from opd_study.models import TinyCausalLM, TinyTransformerConfig

tokenizer = CharacterTokenizer(); splits = generate_tiny_arithmetic(train_rows=8, validation_rows=2, test_rows=2)
batch = collate_examples(splits.train[:2], tokenizer)
config = TinyTransformerConfig(vocab_size=tokenizer.vocab_size, number_of_layers=1,
    hidden_size=32, number_of_heads=4, feed_forward_size=64)
initial = TinyCausalLM(config); teacher = TinyCausalLM(config)
sft_student = copy.deepcopy(initial); kd_student = copy.deepcopy(initial)
teacher_signals = score_teacher(teacher, batch)
sft_output = supervised_fine_tuning_loss(sft_student(batch.token_ids, batch.attention_mask), batch)
kd_output = off_policy_kd_loss(kd_student(batch.token_ids, batch.attention_mask), batch, teacher_signals)
print("same response targets:", int(sft_output.effective_mask.sum()), int(kd_output.effective_mask.sum()))

same response targets: 68 68


In [4]:
from opd_study.utils import model_state_hash

print("initial hashes equal:", model_state_hash(sft_student) == model_state_hash(kd_student))
print("SFT reads hard target IDs; off-policy KD reads teacher distributions on the same fixed prefixes.")

initial hashes equal: True
SFT reads hard target IDs; off-policy KD reads teacher distributions on the same fixed prefixes.


## Checks

In [5]:
assert model_state_hash(sft_student) == model_state_hash(kd_student)
assert int(sft_output.effective_mask.sum()) == int(kd_output.effective_mask.sum())
assert not teacher_signals.logits.requires_grad
print("check passed: initialization, state source, token budget, and teacher detach are auditable")

check passed: initialization, state source, token budget, and teacher detach are auditable


**Exercise (7 min):** print one SFT and KD response mask, then assert that doubling prompt length does not change the effective token budget.

<details><summary>Check</summary>Prompt positions remain false; only response targets count toward budget.</details>

## My recurring mistakes

### M1 — Assuming equal targets imply equal learning

- Wrong: SFT and KD share an answer, so they are identical.
- Why: hard IDs and teacher distributions carry different information.
- Fix: label state source and target representation separately.
- Related check: `test_teacher_is_frozen_and_student_updates`

### M2 — Reporting only optimizer steps for fairness

- Wrong: equalize steps despite different sequence lengths.
- Why: processed response-token counts can differ.
- Fix: compare initial hash, split, steps, and token budget.
- Related check: `test_demo_writes_all_learner_facing_artifacts`

## 60-second summary

1. identify who creates training states
2. compare SFT and KD fairly
3. audit response budgets

## Next Steps

Before the next notebook, rerun the assertions and record one prediction you revised.

### Sources

- [`gkd`](https://arxiv.org/abs/2306.13649v3) · `2306.13649v3` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)